# 5장 실습 — GTFS 시간표 열어 보기

대중교통 시간표는 GTFS 라는 국제 표준으로 배포됩니다.
표 다섯 개를 열어 보고, 한국 데이터에서 걸려 넘어지는 두 곳을 확인합니다.
교재 5장에 대응합니다.

기말 프로젝트에서 여러분이 고른 시군구의 GTFS 를 자르는 것도 여기서 배웁니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 표 다섯 개 (교재 5.1)

In [ ]:
from smartmob.data import describe_feed, load_gtfs

feed = load_gtfs("hanam")
describe_feed(feed)

In [ ]:
for name, table in feed.items():
    print(f"{name:12s} {len(table):>8,}행   {list(table.columns)[:6]}")

표 사이의 연결은 이렇습니다.
노선(`routes`) 하나에 운행(`trips`) 이 여러 개 달리고,
운행 하나가 정류장(`stops`) 을 순서대로 지나며 그 시각이 `stop_times` 에 들어 있습니다.

In [ ]:
feed["stop_times"].head()

## 2. route_type 이 국제 표준과 다릅니다 (교재 5.2)

GTFS 표준에서 `3` 은 버스입니다. 한국 데이터는 여기에 자체 코드를 얹어 씁니다.

In [ ]:
from smartmob.data import KOREAN_ROUTE_TYPE

counts = feed["routes"]["route_type"].astype(int).value_counts().sort_index()
for code, n in counts.items():
    print(f"{code:3d}  {KOREAN_ROUTE_TYPE.get(int(code), '기타'):10s} {n:4d}개 노선")

코드를 국제 표준으로 읽으면 노선 종류를 통째로 잘못 분류합니다.

## 3. 시각이 24시를 넘습니다 (교재 5.3)

새벽 1시에 끝나는 막차는 `25:10:00` 으로 적힙니다.
`datetime` 으로 바로 파싱하면 터집니다.

In [ ]:
from smartmob.data import parse_gtfs_time, seconds_to_gtfs_time

late = feed["stop_times"]["departure_time"].astype(str)
over_24 = late[late.str.slice(0, 2).astype(int) >= 24]

print(f"24시를 넘는 시각 {len(over_24):,}건")
print(over_24.head(5).tolist())

In [ ]:
for text in ["08:30:00", "24:05:00", "25:10:00"]:
    secs = parse_gtfs_time(text)
    print(f"{text}  →  {secs:6,}초  →  다시 {seconds_to_gtfs_time(secs)}")

## 4. 정류장 지도 (교재 5.4)

In [ ]:
import matplotlib.pyplot as plt

stops = feed["stops"]
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(stops["stop_lon"], stops["stop_lat"], s=3, alpha=0.5, color="#4C6EF5")
ax.set_xlabel("경도")
ax.set_ylabel("위도")
ax.set_title(f"하남 GTFS 정류장 {len(stops):,}곳")
plt.tight_layout();

정류장이 하남 경계 밖까지 퍼져 있습니다.
하남을 지나는 노선의 나머지 구간이 함께 들어 있기 때문입니다.
이것이 다음 절의 이유입니다.

## 5. 잘라 낼 때 노선 전체를 남깁니다 (교재 5.5)

경계 안의 정류장만 남기면 버스가 시 경계에서 사라집니다.
경계에 닿는 **노선 전체**를 남겨야 합니다.

In [ ]:
from smartmob.data import clip_to_boundary, list_sigungu, load_sigungu

print(list_sigungu(contains="하남")[:5])

In [ ]:
boundary = load_sigungu("하남시")
clipped = clip_to_boundary(feed, boundary, buffer_m=500)

banner("경계로 자른 뒤")
before, after = describe_feed(feed), describe_feed(clipped)
for key in ["stops", "routes", "trips", "stop_times"]:
    print(f"{key:12s} {before[key]:>8,} → {after[key]:>8,}")

## 6. 빈칸

### 6.1 운행이 가장 많은 노선

`trips` 를 `route_id` 로 묶어 운행 횟수 상위 5개 노선을 찾습니다.
`routes` 와 합쳐 노선 이름(`route_short_name` 또는 `route_long_name`)도 함께 봅니다.

In [ ]:
busiest = None      # 상위 5개 노선 (DataFrame 또는 Series)

banner("빈칸 6.1")
todo("운행이 가장 많은 노선 5개", busiest, fmt=lambda x: f"{len(x)}개")

### 6.2 첫차와 막차

전체 `stop_times` 에서 가장 이른 출발 시각과 가장 늦은 출발 시각을 구합니다.
`parse_gtfs_time` 으로 초로 바꾼 뒤 비교해야 합니다. 문자열로 비교하면 25시가 8시보다 작습니다.

In [ ]:
first_departure = None     # 가장 이른 출발 (초)
last_departure = None      # 가장 늦은 출발 (초)

banner("빈칸 6.2")
todo("첫차", first_departure, fmt=seconds_to_gtfs_time)
todo("막차", last_departure, fmt=seconds_to_gtfs_time)

### 6.3 우리 동네

`list_sigungu()` 에서 여러분이 사는 시군구를 찾아 이름을 적습니다.
기말 프로젝트의 대상지가 될 곳입니다. 인구 20~50만 규모를 권합니다.

지금은 이름만 확인하면 됩니다. 전국 GTFS 를 자르는 것은 프로젝트 6주차에 합니다.

In [ ]:
my_sigungu = None      # 예: "성남시 수정구"

banner("빈칸 6.3")
if my_sigungu:
    info = load_sigungu(my_sigungu)
    print(f"[v] {my_sigungu} 경계를 찾았습니다")
else:
    print("[ ] my_sigungu 를 채우세요. 후보:", list_sigungu()[:8], "...")

## 정리

- GTFS 는 표 다섯 개입니다. `routes` → `trips` → `stop_times` 로 이어집니다
- 한국 데이터의 `route_type` 은 국제 표준과 다릅니다. `KOREAN_ROUTE_TYPE` 을 씁니다
- 시각이 24시를 넘습니다. `parse_gtfs_time` 으로 초로 바꿔 다룹니다
- 경계로 자를 때는 정류장이 아니라 노선 전체를 남깁니다
- 6장 실습에서는 이 시간표 위에서 환승 경로를 직접 찾습니다